# 07 — Exp Weighted Ensemble / 集成方法

**Chapter 26 — File 7 of 7 / 第26章 — 第7个文件（共7个）**

---

## Summary / 总结

This script demonstrates **exponentially decreasing weighted average of models on blobs problem**.

本脚本演示 **exponentially decreasing weighted average of models on blobs problem**。

---
## Step 1 — exponentially decreasing weighted average of models on blobs problem

In [ ]:
from sklearn.datasets import make_blobs
from keras.utils import to_categorical
from keras.models import load_model
from keras.models import clone_model
from matplotlib import pyplot
from numpy import average
from numpy import array
from math import exp

---
## Step 2 — load models from file

In [ ]:
def load_all_models(n_start, n_end):
	all_models = list()
	for epoch in range(n_start, n_end):

---
## Step 3 — define filename for this ensemble

In [ ]:
filename = 'model_' + str(epoch) + '.h5'

---
## Step 4 — load model from file

In [ ]:
model = load_model(filename)

---
## Step 5 — add to list of members

In [ ]:
all_models.append(model)
		print('>loaded %s' % filename)
	return all_models

---
## Step 6 — create a model from the weights of multiple models

In [ ]:
def model_weight_ensemble(members, weights):

---
## Step 7 — determine how many layers need to be averaged

In [ ]:
n_layers = len(members[0].get_weights())

---
## Step 8 — create an set of average model weights

In [ ]:
avg_model_weights = list()
	for layer in range(n_layers):

---
## Step 9 — collect this layer from each model

In [ ]:
layer_weights = array([model.get_weights()[layer] for model in members])

---
## Step 10 — weighted average of weights for this layer

In [ ]:
avg_layer_weights = average(layer_weights, axis=0, weights=weights)

---
## Step 11 — store average layer weights

In [ ]:
avg_model_weights.append(avg_layer_weights)

---
## Step 12 — create a new model with the same structure

In [ ]:
model = clone_model(members[0])

---
## Step 13 — set the weights in the new

In [ ]:
model.set_weights(avg_model_weights)
	model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
	return model

---
## Step 14 — evaluate a specific number of members in an ensemble

In [ ]:
def evaluate_n_members(members, n_members, testX, testy):

---
## Step 15 — select a subset of members

In [ ]:
subset = members[:n_members]

---
## Step 16 — prepare an array of exponentially decreasing weights

In [ ]:
alpha = 2.0
	weights = [exp(-i/alpha) for i in range(1, n_members+1)]

---
## Step 17 — create a new model with the weighted average of all model weights

In [ ]:
model = model_weight_ensemble(subset, weights)

---
## Step 18 — make predictions and evaluate accuracy

In [ ]:
_, test_acc = model.evaluate(testX, testy, verbose=0)
	return test_acc

---
## Step 19 — generate 2d classification dataset

In [ ]:
X, y = make_blobs(n_samples=1100, centers=3, n_features=2, cluster_std=2, random_state=2)

---
## Step 20 — one hot encode output variable

In [ ]:
y = to_categorical(y)

---
## Step 21 — split into train and test

In [ ]:
n_train = 100
trainX, testX = X[:n_train, :], X[n_train:, :]
trainy, testy = y[:n_train], y[n_train:]

---
## Step 22 — load models in order

In [ ]:
members = load_all_models(490, 500)
print('Loaded %d models' % len(members))

---
## Step 23 — reverse loaded models so we build the ensemble with the last models first

In [ ]:
members = list(reversed(members))

---
## Step 24 — evaluate different numbers of ensembles on hold out set

In [ ]:
single_scores, ensemble_scores = list(), list()
for i in range(1, len(members)+1):

---
## Step 25 — evaluate model with i members

In [ ]:
ensemble_score = evaluate_n_members(members, i, testX, testy)

---
## Step 26 — evaluate the i'th model standalone

In [ ]:
_, single_score = members[i-1].evaluate(testX, testy, verbose=0)

---
## Step 27 — summarize this step

In [ ]:
print('> %d: single=%.3f, ensemble=%.3f' % (i, single_score, ensemble_score))
	ensemble_scores.append(ensemble_score)
	single_scores.append(single_score)

---
## Step 28 — plot score vs number of ensemble members

In [ ]:
x_axis = [i for i in range(1, len(members)+1)]
pyplot.plot(x_axis, single_scores, marker='o', linestyle='None')
pyplot.plot(x_axis, ensemble_scores, marker='o')
pyplot.show()

---
## Learning Notes / 学习笔记

- **概念**: exponentially decreasing weighted average of models on blobs problem 是机器学习中的常用技术。  
  *exponentially decreasing weighted average of models on blobs problem is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Exp Weighted Ensemble / 集成方法
# Complete Code / 完整代码
# ===============================

# exponentially decreasing weighted average of models on blobs problem
from sklearn.datasets import make_blobs
from keras.utils import to_categorical
from keras.models import load_model
from keras.models import clone_model
from matplotlib import pyplot
from numpy import average
from numpy import array
from math import exp

# load models from file
def load_all_models(n_start, n_end):
	all_models = list()
	for epoch in range(n_start, n_end):
		# define filename for this ensemble
		filename = 'model_' + str(epoch) + '.h5'
		# load model from file
		model = load_model(filename)
		# add to list of members
		all_models.append(model)
		print('>loaded %s' % filename)
	return all_models

# create a model from the weights of multiple models
def model_weight_ensemble(members, weights):
	# determine how many layers need to be averaged
	n_layers = len(members[0].get_weights())
	# create an set of average model weights
	avg_model_weights = list()
	for layer in range(n_layers):
		# collect this layer from each model
		layer_weights = array([model.get_weights()[layer] for model in members])
		# weighted average of weights for this layer
		avg_layer_weights = average(layer_weights, axis=0, weights=weights)
		# store average layer weights
		avg_model_weights.append(avg_layer_weights)
	# create a new model with the same structure
	model = clone_model(members[0])
	# set the weights in the new
	model.set_weights(avg_model_weights)
	model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
	return model

# evaluate a specific number of members in an ensemble
def evaluate_n_members(members, n_members, testX, testy):
	# select a subset of members
	subset = members[:n_members]
	# prepare an array of exponentially decreasing weights
	alpha = 2.0
	weights = [exp(-i/alpha) for i in range(1, n_members+1)]
	# create a new model with the weighted average of all model weights
	model = model_weight_ensemble(subset, weights)
	# make predictions and evaluate accuracy
	_, test_acc = model.evaluate(testX, testy, verbose=0)
	return test_acc

# generate 2d classification dataset
X, y = make_blobs(n_samples=1100, centers=3, n_features=2, cluster_std=2, random_state=2)
# one hot encode output variable
y = to_categorical(y)
# split into train and test
n_train = 100
trainX, testX = X[:n_train, :], X[n_train:, :]
trainy, testy = y[:n_train], y[n_train:]
# load models in order
members = load_all_models(490, 500)
print('Loaded %d models' % len(members))
# reverse loaded models so we build the ensemble with the last models first
members = list(reversed(members))
# evaluate different numbers of ensembles on hold out set
single_scores, ensemble_scores = list(), list()
for i in range(1, len(members)+1):
	# evaluate model with i members
	ensemble_score = evaluate_n_members(members, i, testX, testy)
	# evaluate the i'th model standalone
	_, single_score = members[i-1].evaluate(testX, testy, verbose=0)
	# summarize this step
	print('> %d: single=%.3f, ensemble=%.3f' % (i, single_score, ensemble_score))
	ensemble_scores.append(ensemble_score)
	single_scores.append(single_score)
# plot score vs number of ensemble members
x_axis = [i for i in range(1, len(members)+1)]
pyplot.plot(x_axis, single_scores, marker='o', linestyle='None')
pyplot.plot(x_axis, ensemble_scores, marker='o')
pyplot.show()